# Week 3 Project 1 — Open-source GenAI use cases with Ollama

This local-first notebook mirrors the section's open-source model workflow. It runs ten reusable GenAI tasks through Ollama, keeps the prompts inspectable, and validates structured output before saving results.

Start Ollama first (`ollama serve`) and pull a small model such as `llama3.2:1b`.


In [ ]:
from pathlib import Path
import json
import re
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "llama3.2:1b"
TIMEOUT_SECONDS = 180


In [ ]:
def ollama_chat(prompt, system="You are a helpful assistant.", temperature=0.2):
    response = requests.post(f"{OLLAMA_URL}/api/chat", json={"model": MODEL, "messages": [{"role": "system", "content": system}, {"role": "user", "content": prompt}], "stream": False, "options": {"temperature": temperature, "num_ctx": 4096, "num_predict": 512}}, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

def extract_json(text):
    candidate = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I | re.S).strip()
    try: return json.loads(candidate)
    except json.JSONDecodeError:
        match = re.search(r"(\{.*\}|\[.*\])", candidate, flags=re.S)
        if not match: raise ValueError(f"The model did not return JSON: {text[:240]}")
        return json.loads(match.group(1))

def ask_json(instruction, schema):
    raw = ollama_chat(f"{instruction}\nReturn only valid JSON matching this schema: {json.dumps(schema)}", system="You return concise, valid JSON and no Markdown fences.", temperature=0)
    return extract_json(raw)


In [ ]:
SAMPLE_TEXT = """Acme's support team met on Tuesday. The team will publish a bilingual FAQ next week,
measure unanswered tickets, and review the results with Sara on 15 May. Customers asked for
faster replies and clearer refund instructions. The current backlog is 184 tickets."""
print(ollama_chat("Summarize this in one sentence:\n" + SAMPLE_TEXT))


In [ ]:
TASKS = {
    "summary": "Summarize the text in three short bullet points.",
    "translation": "Translate the text into Arabic while preserving names and dates.",
    "classification": "Classify the text as one of: support, sales, finance, engineering, other.",
    "sentiment": "Return the sentiment as positive, neutral, or negative with a one-sentence reason.",
    "keywords": "Return the five most useful keywords as a JSON array of strings.",
    "rewrite": "Rewrite the text as a clear, friendly update for a manager.",
    "qa": "Answer this question from the text: What will the team review on 15 May?",
    "agenda": "Turn the text into a three-item meeting agenda.",
    "comparison": "State one risk and one opportunity implied by the text.",
    "extraction": "Extract the named people, dates, quantities, and commitments.",
}
JSON_TASKS = {"keywords", "extraction"}
def run_task(name, text=SAMPLE_TEXT):
    instruction = TASKS[name] + "\n\nTEXT:\n" + text
    if name == "classification": return ask_json(instruction, {"label": "support|sales|finance|engineering|other", "reason": "string"})
    if name == "sentiment": return ask_json(instruction, {"label": "positive|neutral|negative", "reason": "string"})
    if name in JSON_TASKS:
        schema = {"keywords": ["string"]} if name == "keywords" else {"people": ["string"], "dates": ["string"], "quantities": ["string"], "commitments": ["string"]}
        return ask_json(instruction, schema)
    return ollama_chat(instruction)


In [ ]:
results = {}
for name in TASKS:
    try:
        results[name] = run_task(name)
        print(f"\n--- {name} ---\n{results[name]}")
    except Exception as exc:
        results[name] = {"error": str(exc)}
        print(f"{name}: {exc}")


In [ ]:
result_dir = Path("results")
result_dir.mkdir(exist_ok=True)
(result_dir / "week3_open_source_use_cases.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(results)} task results to {result_dir / 'week3_open_source_use_cases.json'}")


## Optional Gradio wrapper

The core functions above are useful in scripts and tests. Set `RUN_UI = True` only when you want to launch a local browser UI. Keep the flag false on a remote notebook until you have an SSM tunnel.


In [ ]:
import gradio as gr
RUN_UI = False
def ui_task(task_name, text):
    value = run_task(task_name, text)
    return json.dumps(value, ensure_ascii=False, indent=2) if task_name in JSON_TASKS | {"classification", "sentiment"} else str(value)
if RUN_UI:
    gr.Interface(fn=ui_task, inputs=[gr.Dropdown(list(TASKS), value="summary", label="Task"), gr.Textbox(value=SAMPLE_TEXT, lines=8, label="Text")], outputs=gr.Textbox(lines=12, label="Ollama result"), title="Week 3 open-source GenAI lab").launch()
